In [3]:
import os
import pickle
import torch
import faiss
import numpy as np
from tqdm import tqdm
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.docstore.in_memory import InMemoryDocstore

In [4]:
EMBEDDING_DIM = 384  # MiniLM output dimension
N_CLUSTERS = 256  # number of IVF clusters
BATCH_SIZE = 5000
CHECKPOINT_EVERY = 10  # save every N batches

In [5]:
# load chunks
with open("/content/drive/MyDrive/nepali-news-rag-chatbot/data/chunks/nepali_news_chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

print(f"Loaded {len(chunks)} chunks")

Loaded 593779 chunks


In [6]:
# load embedding model
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model_kwargs = {"device": "cuda" if torch.cuda.is_available() else "cpu"}
encode_kwargs = {"normalize_embeddings": True}

In [7]:
hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

/tmp/ipython-input-3535919855.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
# embed a training sample (raw vectors)
sample_texts = [c.page_content for c in chunks[:50_000]]
# sample_vectors = np.array(hf_embeddings.embed_documents(sample_texts), dtype=np.float32)
# print(f"Training sample shape: {sample_vectors.shape}")


all_vectors = []

for i in tqdm(range(0, len(sample_texts), BATCH_SIZE), desc="Embedding training sample"):
    batch = sample_texts[i : i + BATCH_SIZE]
    batch_vectors = hf_embeddings.embed_documents(batch)
    all_vectors.extend(batch_vectors)

sample_vectors = np.array(all_vectors, dtype=np.float32)
print(f"Training sample shape: {sample_vectors.shape}")

Embedding training sample: 100%|██████████| 10/10 [02:14<00:00, 13.47s/it]


Training sample shape: (50000, 384)


In [9]:
# build and train the IVF index
quantizer = faiss.IndexFlatL2(EMBEDDING_DIM)
index = faiss.IndexIVFFlat(quantizer, EMBEDDING_DIM, N_CLUSTERS, faiss.METRIC_L2)

In [10]:
# train the index
index.train(sample_vectors)
print(f"IVF index trained. {index.ntotal} vectors in index (training only).")

IVF index trained. 0 vectors in index (training only).


In [11]:
# wrap the trained index in a LangChain FAISS vectorstore
vectorstore = FAISS(
    embedding_function=hf_embeddings.embed_query,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)
print("Vectorstore ready with trained IVF index.")

Vectorstore ready with trained IVF index.


In [12]:
# create directories if they don't exist
checkpoint_dir = "/content/drive/MyDrive/nepali-news-rag-chatbot/faiss_index_nepali_checkpoint"
final_index_dir = "/content/drive/MyDrive/nepali-news-rag-chatbot/faiss_index_nepali"
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(final_index_dir, exist_ok=True)

In [13]:
# batch-insert all chunks with periodic checkpoints
num_batches = (len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE

for i in tqdm(range(0, len(chunks), BATCH_SIZE), total=num_batches, desc="Embedding batches"):
    batch = chunks[i : i + BATCH_SIZE]
    vectorstore.add_documents(batch)

    batch_num = (i // BATCH_SIZE) + 1

    if batch_num % CHECKPOINT_EVERY == 0:
        vectorstore.save_local(checkpoint_dir)
        tqdm.write(f"Batch {batch_num:>4d} | chunks {i:>7d} → {i + len(batch):>7d} | Checkpoint saved | total: {index.ntotal}")
    else:
        tqdm.write(f"Batch {batch_num:>4d} | chunks {i:>7d} → {i + len(batch):>7d} | total: {index.ntotal}")

print(f"\nDone inserting. Final index size: {index.ntotal} vectors.")

# save the final index
vectorstore.save_local(final_index_dir)
print(f"Final FAISS index saved to {final_index_dir}")

Embedding batches:   1%|          | 1/119 [01:05<2:09:03, 65.62s/it]

Batch    1 | chunks       0 →    5000 | total: 5000


Embedding batches:   2%|▏         | 2/119 [02:08<2:04:37, 63.91s/it]

Batch    2 | chunks    5000 →   10000 | total: 10000


Embedding batches:   3%|▎         | 3/119 [03:09<2:01:28, 62.84s/it]

...

Batch  117 | chunks  580000 →  585000 | total: 585000


Embedding batches:  99%|█████████▉| 118/119 [2:21:03<01:14, 74.48s/it]

Batch  118 | chunks  585000 →  590000 | total: 590000


Embedding batches: 100%|██████████| 119/119 [2:21:58<00:00, 71.59s/it]


Batch  119 | chunks  590000 →  593779 | total: 593779

Done inserting. Final index size: 593779 vectors.
Final FAISS index saved to /content/drive/MyDrive/nepali-news-rag-chatbot/faiss_index_nepali
